In [ ]:
import os
os.environ["HF_HUB_VERBOSITY"] = "error"

import json
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# -------------------------------------------------------------------
# 1. SETUP MODEL (Run this once per instance)
# -------------------------------------------------------------------
model_name = "Qwen3-Embedding-4B"
model_dir = f"../embed-model/{model_name}"
os.makedirs(model_dir, exist_ok=True)
os.environ["HF_HOME"] = model_dir

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️ Using device: {device}")

# Load the Qwen3 model
model = SentenceTransformer(
    f"Qwen/{model_name}", 
    device=device,
    cache_folder=model_dir, 
    model_kwargs={"torch_dtype": torch.float16} 
)

# -------------------------------------------------------------------
# 2. HELPER FUNCTION: JSON FORMATTING
# -------------------------------------------------------------------
def format_row_pois_as_json(row, max_pois=10):
    """
    Format POIs for a single row as JSON string.
    """
    poi_dict = {}
    has_any_poi = False
    
    for i in range(1, max_pois + 1):
        name_col = f"POI_{i}_name"
        type_col = f"POI_{i}_type"
        lat_col = f"POI_{i}_lat"
        lng_col = f"POI_{i}_lng"
        
        if name_col in row.index and pd.notna(row[name_col]) and str(row[name_col]).strip() != '':
            poi_info = {}
            
            poi_info['name'] = str(row[name_col]).strip()
            
            if type_col in row.index and pd.notna(row[type_col]):
                poi_info['type'] = str(row[type_col]).strip()
            if lat_col in row.index and pd.notna(row[lat_col]):
                poi_info['lat'] = float(row[lat_col])
            if lng_col in row.index and pd.notna(row[lng_col]):
                poi_info['lng'] = float(row[lng_col])
            
            dist_col = f"POI_{i}_dist(m)"
            if dist_col in row.index and pd.notna(row[dist_col]):
                poi_info['dist(m)'] = float(row[dist_col])
            
            dir_col = f"POI_{i}_dir(deg)"
            if dir_col in row.index and pd.notna(row[dir_col]):
                poi_info['dir(deg)'] = float(row[dir_col])
            
            poi_dict[f"POI_{i}"] = poi_info
            has_any_poi = True
    
    if not has_any_poi:
        return "No POIs found within the given radius."
    
    return json.dumps(poi_dict, ensure_ascii=False)


# -------------------------------------------------------------------
# 3. CORE FUNCTION: PARALLEL-SAFE CHUNK GENERATION
# -------------------------------------------------------------------
def save_poi_embed_in_chunks(input_data, dataset, place, chunk_size=5000, batch_size=32, max_pois=10, start_chunk=0, end_chunk=None):
    """
    Processes data in chunks, strictly loading only the needed rows into RAM to prevent OOM errors.
    Includes tqdm progress bars for BOTH JSON formatting and GPU embedding.
    """
    embedding_dir = f"../Processed Datasets/{dataset}/Embedding"
    chunks_dir = os.path.join(embedding_dir, "chunks")
    json_chunks_dir = os.path.join(chunks_dir, "json_chunks")
    os.makedirs(json_chunks_dir, exist_ok=True)
    
    # --- OPTIMIZATION 1: The Parallel RAM Trap ---
    is_dataframe = isinstance(input_data, pd.DataFrame)
    if is_dataframe:
        total_rows = len(input_data)
    else:
        # Fast way to count total rows in a massive CSV without loading it into RAM
        print("📊 Counting total rows in CSV (RAM-safe)...")
        with open(input_data, 'r', encoding='utf-8') as f:
            total_rows = sum(1 for _ in f) - 1  # subtract header
            
    total_chunks = (total_rows + chunk_size - 1) // chunk_size

    if end_chunk is None or end_chunk > total_chunks:
        end_chunk = total_chunks

    print(f"🚀 Total dataset size: {total_rows} rows ({total_chunks} total chunks).")
    print(f"🎯 This instance is processing chunks {start_chunk + 1} through {end_chunk}...")

    for chunk_id in range(start_chunk, end_chunk):
        npy_path = os.path.join(chunks_dir, f'{dataset}-{place}-poi-embed-part{chunk_id+1}.npy')
        json_chunk_path = os.path.join(json_chunks_dir, f'{dataset}-{place}-json-part{chunk_id+1}.json')

        if os.path.exists(npy_path):
            print(f"⏭️ Skipping chunk {chunk_id+1}/{total_chunks} (Embedding already exists)")
            continue

        start_row = chunk_id * chunk_size
        end_row = min(start_row + chunk_size, total_rows)
        
        if os.path.exists(json_chunk_path):
            print(f"📂 Loading existing JSON chunk {chunk_id+1}...")
            with open(json_chunk_path, 'r', encoding='utf-8') as f:
                chunk_texts = json.load(f)
        else:
            print(f"📝 Loading rows {start_row} to {end_row}...")
            
            # Load strictly the required chunk to save massive amounts of RAM
            if is_dataframe:
                chunk_df = input_data.iloc[start_row:end_row]
            else:
                chunk_df = pd.read_csv(
                    input_data, 
                    skiprows=range(1, start_row + 1), #type: ignore # Skips previous data rows, keeps the header (row 0)
                    nrows=chunk_size, 
                    low_memory=False) #type: ignore
 
            # --- OPTIMIZATION 2: The `progress_apply()` TQDM Bar ---
            tqdm.pandas(desc=f"Formatting Chunk {chunk_id+1}/{total_chunks}")
            chunk_texts = chunk_df.progress_apply(lambda row: format_row_pois_as_json(row, max_pois), axis=1).tolist()
            
            with open(json_chunk_path, 'w', encoding='utf-8') as f:
                json.dump(chunk_texts, f, ensure_ascii=False)
                
            # Clear chunk dataframe from memory
            del chunk_df

        # --- EMBED ON GPU ---
        print(f"🧠 Encoding Chunk {chunk_id+1}/{total_chunks} on GPU...")
        chunk_embeds = model.encode(chunk_texts, batch_size=batch_size, show_progress_bar=True)
        chunk_arr = np.array(chunk_embeds, dtype=np.float32)
        
        np.save(npy_path, chunk_arr)
        print(f"✅ Saved embedding chunk {chunk_id+1}/{total_chunks}\n")
        
        # Aggressive memory cleanup
        del chunk_texts, chunk_embeds, chunk_arr
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# -------------------------------------------------------------------
# 4. FINALIZATION: RAM-SAFE CONCATENATION
# -------------------------------------------------------------------
def concat_chunks(dataset, place, output_version="v2"):
    """
    Concatenate all chunked .npy files. Uses pre-allocation to prevent RAM from doubling/spiking.
    """
    embedding_dir = f"../Processed Datasets/{dataset}/Embedding"
    chunks_dir = os.path.join(embedding_dir, "chunks")
    
    if not os.path.exists(chunks_dir):
        print(f"⚠️ Chunks directory not found: {chunks_dir}")
        return None
    
    files = [f for f in os.listdir(chunks_dir) if f.startswith(f"{dataset}-{place}-poi-embed-part") and f.endswith(".npy")]

    if not files:
        print("⚠️ No chunk files found to concatenate.")
        return None

    # Sort numerically (part1, part2... part10) instead of alphabetically
    files.sort(key=lambda x: int(x.split('-part')[-1].split('.npy')[0]))

    print(f"📦 Loading {len(files)} chunk files from {chunks_dir}...")
    
    # --- OPTIMIZATION 3: The `np.vstack` Memory Spike ---
    # Load just the first chunk to get the embedding dimension
    first_chunk = np.load(os.path.join(chunks_dir, files[0]))
    embedding_dim = first_chunk.shape[1]
    
    print("📏 Calculating total array size...")
    # mmap_mode='r' allows us to read the array shape without actually loading it into RAM!
    total_rows = sum(np.load(os.path.join(chunks_dir, f), mmap_mode='r').shape[0] for f in files)
    
    print(f"🏗️ Pre-allocating memory for {total_rows} rows...")
    final_arr = np.empty((total_rows, embedding_dim), dtype=np.float32)
    
    current_idx = 0
    for f in tqdm(files, desc="Concatenating Chunks"):
        chunk = np.load(os.path.join(chunks_dir, f))
        chunk_len = len(chunk)
        final_arr[current_idx : current_idx + chunk_len] = chunk
        current_idx += chunk_len

    final_path = os.path.join(embedding_dir, f'{dataset}-{place}-poi-embed-{output_version}.npy')
    
    print("💾 Saving final massive array to disk...")
    np.save(final_path, final_arr)
    print(f"✅ Saved final embeddings: {final_path}")
    print(f"   Shape: {final_arr.shape}")

    return final_arr, files

# -------------------------------------------------------------------
# 5. EXECUTION EXAMPLE
# -------------------------------------------------------------------
if __name__ == "__main__":
    place = "Beijing"
    dataset = "geolife"
    v = "-v2"
    input_csv = f"../Processed Datasets/{dataset}/{dataset}-{place}-POIs{v}.csv"

    # Example: Run chunks 1 through 10
    save_poi_embed_in_chunks(
        input_data=input_csv,
        dataset=dataset,
        place=place,
        chunk_size=100000,   # Safely increased to process 100k rows at a time
        batch_size=64,
        max_pois=10,
        start_chunk=0,      # Change this in different terminal instances to parallelize
        end_chunk=10        # e.g., Terminal 2: start_chunk=10, end_chunk=20
    )

⚙️ Using device: cuda


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

📊 Counting total rows in CSV (RAM-safe)...
🚀 Total dataset size: 5941134 rows (60 total chunks).
🎯 This instance is processing chunks 1 through 10...
📝 Loading rows 0 to 100000...


Formatting Chunk 1/60: 100%|██████████| 100000/100000 [00:26<00:00, 3718.33it/s]


🧠 Encoding Chunk 1/60 on GPU...


...

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

✅ Saved embedding chunk 4/60

📝 Loading rows 400000 to 500000...


Formatting Chunk 5/60: 100%|██████████| 100000/100000 [00:27<00:00, 3574.45it/s]


🧠 Encoding Chunk 5/60 on GPU...


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

In [ ]:
# # Run this ONLY when all chunks are completely finished
# final_arr, files = concat_chunks(
#     dataset=dataset,
#     place=place,
#     output_version=v
# )